# Entrenar un detector con tus datos

Si el detector no encuentra bien las aves de tus videos, puedes ajustarlo con imágenes parecidas a las tuyas. Este notebook prepara
tus datos, entrena y compara el resultado con el detector actual. Necesita una GPU; el tiempo depende de los datos: con unas 5.000
imágenes, 20 épocas y 6 GB de GPU, unas 7 horas.

El detector tiene **una sola clase**: todas las etiquetas se unifican como colibrí.

In [ ]:
import pandas as pd
from IPython.display import display
from colibri.config import CARPETAS, DETECTOR
from colibri.entrenamiento import comparar_detectores, entrenar, preparar_datos

## 1. Tus datos

Cada dataset puede ser una carpeta o un proyecto de Roboflow, y puedes combinar varios.

**Carpeta.** Colócala en `datasets/` con este formato. `test` es opcional, y cada línea de etiqueta es `clase cx cy ancho alto`, normalizados de 0 a 1;
también valen polígonos. Una imagen sin etiqueta, o con la etiqueta vacía, cuenta como imagen sin ave: también ayudan a evitar falsas detecciones.

```
datasets/mis_fotos/
├─ train/images/*.jpg    train/labels/*.txt
├─ valid/images/*.jpg    valid/labels/*.txt
└─ test/images/*.jpg     test/labels/*.txt
```

**Roboflow.** Copia `.env.example` como `.env`, pon tu `ROBOFLOW_API_KEY`, y usa los datos del proyecto.

```python
DATASETS = {
    "mis_fotos": {"carpeta": "mis_fotos"},
    "mi_proyecto": {"workspace": "mi-espacio", "project": "mi-proyecto", "version": 1},
    "mis_videos": {"carpeta": "cuadros_de_mis_videos", "repetir": 3},
}
```

`"repetir": 3` repite ese `train` tres veces. Es útil cuando tienes pocas imágenes de tu caso frente a un dataset grande.

**Evita la fuga de información.** Si tus imágenes son cuadros de un video, no las repartas al azar entre train, valid y test: los cuadros vecinos
son casi iguales y las métricas saldrían infladas. Reparte por video, o al menos por tramos separados. Y deja en test videos que el modelo no haya visto.

In [ ]:
NOMBRE_MODELO = "mi_detector"
DATASETS = {
}

DATA_YAML, TABLA = preparar_datos(NOMBRE_MODELO, DATASETS)
display(TABLA)

## 2. Entrena

- `PESOS_INICIALES`: `DETECTOR` ajusta el detector actual y es lo recomendado con pocos datos. `"yolo26n.pt"` empieza desde un modelo general.
- `PARAMETROS`: `epochs` es el número de pasadas por los datos. `batch=0.70` usa el 70 % de la memoria de la GPU.

Si el entrenamiento se corta, vuelve a ejecutar la celda: continúa donde iba.

In [ ]:
PESOS_INICIALES = DETECTOR
PARAMETROS = dict(epochs=20, patience=8, imgsz=640, warmup_epochs=1, lr0=0.002, optimizer="SGD", cache="disk", workers=3, batch=0.70, seed=0)

SALIDA = CARPETAS["salidas"] / "entrenamiento"
MEJOR = entrenar(NOMBRE_MODELO, PESOS_INICIALES, DATA_YAML, SALIDA, **PARAMETROS)
print("Tu detector:", MEJOR)

## 3. Compara con el detector actual

Ambos se evalúan con las mismas imágenes: las de `test`, o las de `valid` si no hay `test`. Mira sobre todo `mAP50` y `recall`.

**Esta comparación solo vale para tus datos.** Si el test sale de los mismos videos que train, dice poco. La prueba que importa es
ver cómo se comporta en videos nuevos.

In [ ]:
COMPARACION = comparar_detectores({"actual": DETECTOR, NOMBRE_MODELO: MEJOR}, DATA_YAML, SALIDA)
COMPARACION.to_csv(SALIDA / NOMBRE_MODELO / "comparacion.csv", index=False)
display(COMPARACION.round(3))

## 4. Usa tu detector

Pega la ruta que aparece arriba en `DETECTOR_A_USAR` de `trayectorias.ipynb`. Para que sea el de siempre, ponla en `detector` de `config.yaml`.

Si crees que es mejor que el publicado y quieres compartirlo, sigue `modelos/README.md`.